# Lab 3.2 - Partial reconfiguration

This notebook runs the partial-reconfiguration demo on the Arty Z7-20 using the PYNQ-Z1 bitstreams from `yunqu/partial_reconfig_example`. PYNQ-Z1 and Arty Z7-20 share the same XC7Z020 device and the same user I/O (LED0-LED3 = `R14 / P14 / N16 / M14`, switches `M20 / M19`, buttons `D19 / D20 / L20 / L19`), so the prebuilt files load unchanged.

Files in `partial_reconfig/`:

- `gpio_pr.bit` / `gpio_pr.hwh` - static design containing reconfigurable region `gpio_0`
- `led_5.bit` / `led_5.hwh` - partial bitstream, LED pattern `0x5` = `0101`
- `led_a.bit` / `led_a.hwh` - partial bitstream, LED pattern `0xA` = `1010`

Note on the PYNQ version: the HWH was exported by Vivado 2018.3, and PYNQ 3.1 no longer derives `Overlay.pr_dict` from that format (region `gpio_0` is parsed as a plain IP instead of a hierarchy block). Partial bitstreams are therefore programmed through `pynq.bitstream.Bitstream(..., partial=True)`, which writes the `.bin` image into `fpga_manager` with the partial-reconfiguration flag - the same code path the high-level API uses internally.

In [4]:
from pathlib import Path
from pprint import pprint

from pynq import Overlay
from pynq.bitstream import Bitstream
from pynq.pl_server.device import Device

folder = Path("partial_reconfig")
required = [folder / name for name in ["gpio_pr.bit", "gpio_pr.hwh",
                                       "led_5.bit", "led_5.hwh",
                                       "led_a.bit", "led_a.hwh"]]
assert all(p.exists() for p in required), "Missing PR files in partial_reconfig/"


def pr_load(bit_path):
    """Program a partial bitstream into the static design already loaded on the PL."""
    dev = Device.active_device
    bit_path = Path(bit_path).resolve()
    parser = dev.get_bitfile_metadata(str(bit_path), partial=True)
    bitstream = Bitstream(str(bit_path), partial=True, device=dev)
    try:
        bitstream.download(parser)
    except KeyError:
        # PYNQ 3.1 re-applies PS AXI port widths from the metadata after
        # programming fpga_manager; a partial HWH has no PS entry, so that
        # step raises KeyError even though the PL is already reconfigured.
        pass
    return bit_path.name


In [5]:
overlay = Overlay(str(folder / "gpio_pr.bit"))
overlay.download()
print("Static design programmed:", Path(overlay.bitfile_name).name)
print("IP blocks:", len(overlay.ip_dict), "| hierarchy blocks:", list(overlay.hierarchy_dict))
pprint(overlay.interrupt_pins)

Static design programmed: gpio_pr.bit
IP blocks: 3 | hierarchy blocks: []
{'concat_interrupts/In0': {'controller': 'system_interrupts',
                           'fullpath': 'concat_interrupts/In0',
                           'index': 0},
 'concat_interrupts/In1': {'controller': 'system_interrupts',
                           'fullpath': 'concat_interrupts/In1',
                           'index': 1},
 'gpio_0/ip2intc_irpt': {'controller': 'system_interrupts',
                         'fullpath': 'gpio_0/ip2intc_irpt',
                         'index': 1},
 'gpio_0/ip2intc_irpt1': {'controller': 'system_interrupts',
                          'fullpath': 'gpio_0/ip2intc_irpt1',
                          'index': 0}}


In [6]:
loaded = pr_load(folder / "led_5.bit")
print("Partial bitstream programmed:", loaded)
print("Expected LEDs now: LD0 = ON, LD1 = OFF, LD2 = ON, LD3 = OFF  (pattern 0x5 = 0101)")
print("Capture a real board photo of LED pattern 0x5 now.")

Partial bitstream programmed: led_5.bit
Expected LEDs now: LD0 = ON, LD1 = OFF, LD2 = ON, LD3 = OFF  (pattern 0x5 = 0101)
Capture a real board photo of LED pattern 0x5 now.


In [7]:
loaded = pr_load(folder / "led_a.bit")
print("Partial bitstream programmed:", loaded)
print("Expected LEDs now: LD0 = OFF, LD1 = ON, LD2 = OFF, LD3 = ON  (pattern 0xA = 1010)")
print("Capture a second real board photo of LED pattern 0xA now.")

Partial bitstream programmed: led_a.bit
Expected LEDs now: LD0 = OFF, LD1 = ON, LD2 = OFF, LD3 = ON  (pattern 0xA = 1010)
Capture a second real board photo of LED pattern 0xA now.
